In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from tensorflow.keras.models import Model
from torchvision import datasets, transforms, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D,Dense, BatchNormalization 
from tensorflow.keras.layers import Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
data_dir = "/kaggle/input/datasets/afridirahman/brain-stroke-ct-image-dataset/Brain_Data_Organised"


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])


In [ ]:
dataset = datasets.ImageFolder(data_dir, transform=transform)

print("Classes:", dataset.classes)

labels = np.array([dataset.samples[i][1] for i in range(len(dataset))])
print("Class distribution:", np.bincount(labels))


In [ ]:
train_idx, temp_idx = train_test_split(
    np.arange(len(labels)),
    test_size=0.3,
    stratify=labels,
    random_state=42
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=labels[temp_idx],
    random_state=42
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))


In [ ]:
import tensorflow.keras.backend as K

def focal_loss(gamma=2., alpha=0.75):
    def loss(y_true, y_pred):
        y_pred = K.clip(y_pred, 1e-7, 1 - 1e-7)
        pt = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        return -K.mean(alpha * K.pow(1 - pt, gamma) * K.log(pt))
    return loss

In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))

# Train full network
for layer in base_model.layers:
    layer.trainable = True

x = base_model.output
x = GlobalAveragePooling2D()(x)

# Dense block
x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)

output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.00005),
    loss=focal_loss(),
    metrics=['accuracy']
)

model.summary()

In [ ]:
class DenseNet121_Model(nn.Module):
    def __init__(self):
        super(DenseNet121_Model, self).__init__()
        
        self.backbone = models.densenet121(weights="IMAGENET1K_V1")
        
        num_features = self.backbone.classifier.in_features
        
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, 1)
        )

    def forward(self, x):
        return self.backbone(x)


model = DenseNet121_Model().to(device)


In [ ]:
num_normal = np.sum(labels == 0)
num_stroke = np.sum(labels == 1)

pos_weight = torch.tensor([num_normal / num_stroke]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=5e-5)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5
)


In [ ]:
history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}

epochs = 30

for epoch in range(epochs):

    # TRAIN
    model.train()
    train_loss = 0
    train_preds, train_labels = [], []

    for images, labels_batch in train_loader:
        
        images = images.to(device)
        labels_batch = labels_batch.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(images)
        outputs = torch.clamp(outputs, -50, 50)

        loss = criterion(outputs, labels_batch)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        train_preds.extend(preds.cpu().numpy().ravel())
        train_labels.extend(labels_batch.cpu().numpy().ravel())

    train_acc = accuracy_score(train_labels, train_preds)

    # VALIDATION
    model.eval()
    val_loss = 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for images, labels_batch in val_loader:

            images = images.to(device)
            labels_batch = labels_batch.float().unsqueeze(1).to(device)

            outputs = model(images)
            outputs = torch.clamp(outputs, -50, 50)

            loss = criterion(outputs, labels_batch)
            val_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            val_preds.extend(preds.cpu().numpy().ravel())
            val_labels.extend(labels_batch.cpu().numpy().ravel())

    val_acc = accuracy_score(val_labels, val_preds)

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss/len(train_loader))
    history["val_loss"].append(val_loss/len(val_loader))
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f}")


In [ ]:
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels_batch in test_loader:

        images = images.to(device)
        outputs = model(images)
        outputs = torch.clamp(outputs, -50, 50)

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        all_preds.extend(preds.cpu().numpy().ravel())
        all_labels.extend(labels_batch.numpy())
        all_probs.extend(probs.cpu().numpy().ravel())

print("\nCLASSIFICATION REPORT:\n")
print(classification_report(all_labels, all_preds, target_names=["Normal","Stroke"]))

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("Precision:", precision_score(all_labels, all_preds))
print("Recall:", recall_score(all_labels, all_preds))
print("F1 Score:", f1_score(all_labels, all_preds))
print("AUC:", roc_auc_score(all_labels, all_probs))


In [ ]:
cm = confusion_matrix(all_labels, all_preds)

sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=["Normal","Stroke"],
            yticklabels=["Normal","Stroke"],
            cmap="Blues")

plt.title("Confusion Matrix")
plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(all_labels, all_probs)

plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()


In [ ]:
model.eval()

num_images = 6
count = 0

plt.figure(figsize=(12, 8))

with torch.no_grad():
    for images, labels_batch in test_loader:

        images = images.to(device)
        outputs = model(images)
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        for i in range(images.size(0)):

            img = images[i].cpu().permute(1,2,0).numpy()
            img = img * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])
            img = np.clip(img, 0, 1)

            true_label = "Stroke" if labels_batch[i] == 1 else "Normal"
            pred_label = "Stroke" if preds[i] == 1 else "Normal"
            confidence = probs[i].item()

            plt.subplot(2, 3, count+1)
            plt.imshow(img)
            plt.title(f"True: {true_label}\nPred: {pred_label} ({confidence:.2f})")
            plt.axis("off")

            count += 1

            if count == num_images:
                break
        if count == num_images:
            break

plt.tight_layout()
plt.show()


In [ ]:
class DenseNet121_Model(nn.Module):
    def __init__(self):
        super(DenseNet121_Model, self).__init__()

        backbone = models.densenet121(weights="IMAGENET1K_V1")

        self.features = backbone.features
        self.pool = nn.AdaptiveAvgPool2d((1,1))

        num_features = backbone.classifier.in_features

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, 1)
        )

    def forward(self, x):
        features = self.features(x)

        # Clone to avoid inplace issue
        features = features.clone()

        out = torch.relu(features)
        out = self.pool(out)
        out = torch.flatten(out, 1)
        out = self.classifier(out)

        return out


model = DenseNet121_Model().to(device)


In [ ]:
fold_results = {
    "accuracy": [],
    "recall": [],
    "f1": [],
    "auc": []
}

for fold, (train_idx, val_idx) in enumerate(
        kfold.split(np.arange(len(dataset)), labels)):

    print(f"\n===== FOLD {fold+1} =====")

    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

    model = DenseNet121_Model().to(device)

    # Weighted loss per fold
    num_normal = np.sum(labels[train_idx] == 0)
    num_stroke = np.sum(labels[train_idx] == 1)

    pos_weight = torch.tensor([num_normal / num_stroke]).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=5e-5)

    epochs = 10   # increase to 20 if needed

    # ---- Training ----
    for epoch in range(epochs):

        model.train()

        for images, labels_batch in train_loader:

            images = images.to(device)
            labels_batch = labels_batch.float().unsqueeze(1).to(device)

            optimizer.zero_grad()

            outputs = model(images)
            outputs = torch.clamp(outputs, -50, 50)

            loss = criterion(outputs, labels_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

    # ---- Validation ----
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels_batch in val_loader:

            images = images.to(device)

            outputs = model(images)
            outputs = torch.clamp(outputs, -50, 50)

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            all_preds.extend(preds.cpu().numpy().ravel())
            all_labels.extend(labels_batch.numpy())
            all_probs.extend(probs.cpu().numpy().ravel())

    acc = accuracy_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)

    fold_results["accuracy"].append(acc)
    fold_results["recall"].append(rec)
    fold_results["f1"].append(f1)
    fold_results["auc"].append(auc)

    print(f"Accuracy: {acc:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1: {f1:.4f}")
    print(f"AUC: {auc:.4f}")
